In [3]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon

import sys,os
import rasterio
from rasterio import features
from rasterio.crs import CRS

from rasterio.transform import Affine


In [4]:
#读取矢量文件 → 检查目标字段并规范化类型 → 计算总体边界与输出栅格的像素行列数 → 构建仿射变换和 metadata → 建立 GeoTIFF 并把每个 polygon 按属性值“烧录”到栅格（未被覆盖的像元赋 0）→ 写盘。
class Polygon2Raster:
    def polygon_to_raster(self,shp,raster,pixel,field,code=4326):
        '''
        矢量转栅格
        :param shp: 输入矢量全路径，字符串，无默认值
        :param raster: 输出栅格全路径，字符串，无默认值
        :param pixel: 像元大小，单位必须与矢量数据的坐标系单位一致
        :param field: 用于栅格化时赋值的字段名（shapefile中的属性列明）
        :param Code: 输出坐标系代码，默认为4326
        :return: None
        '''

        # 判断字段是否存在
        shapefile = gpd.read_file(shp)
        if not field in shapefile.columns:
            raise Exception ('输出字段不存在')
        # 判断数据类型，这一步是为了与 rasterio 的 dtype 对应，确保写入 tif 时数据类型一致
        f_type = shapefile.dtypes.get(field)
        if 'int' in str(f_type):
            shapefile[field] = shapefile[field].astype('int16')
            dtype = 'int16'
        elif 'float' in str(f_type):
            shapefile[field] = shapefile[field].astype('float32')
            dtype = 'float32'
        else:
            raise Exception ('输入字段数据类型为{}，无法进行栅格化操作'.format(f_type))

        bound = shapefile.bounds  #返回一个 DataFrame，包含每个要素的 minx, miny, maxx, maxy 四列（即每个 geometry 的边界框
        #计算输出栅格的列数（width）与行数（height）
        width = int((bound.get('maxx').max()-bound.get('minx').min())/pixel)
        height = int((bound.get('maxy').max()-bound.get('miny').min())/pixel)
        #该 transform 表示栅格的左上角定位为 (minx, maxy)，每个像元大小为 pixel × pixel（x 正向、y 负向）
        transform = Affine(pixel, 0.0, bound.get('minx').min(),
               0.0, -pixel, bound.get('maxy').max())
        
        #构造用于 rasterio.open(..., 'w') 的元数据字典 meta
        meta = {'driver': 'GTiff',  #输出格式为 GeoTIFF
                'dtype': dtype,     #数据类型（之前根据字段选择 'int16' 或 'float32'）
                'nodata': 0,        #设置无数据值为 0（注意：若 0 是有效值，这会冲突——需谨慎选择 nodata）
                'width': width,
                'height': height,   #像元列数与行数
                'count': 1,         #波段数（单波段
                'crs': CRS.from_epsg(code),    #坐标参考系，用 CRS.from_epsg(code)（rasterio.crs）创建 CRS 对象
                'transform': transform}
        #重要：这里只是把 CRS 设为 code（例如 4326），但并未把 shapefile 的几何重新投影到该 CRS（如果两者 CRS 不同会导致错位）。

        #用 rasterio.open 以读写模式（'w+'）创建/打开输出栅格文件（路径 raster），并把 meta 传进去以创建文件头信息。out 是打开的 DatasetWriter 对象
        with rasterio.open(raster, 'w+', **meta) as out:
            #读取文件第 1 波段到 numpy 数组 out_arr。在新创建的文件上这通常得到一个被 nodata 或 0 填充的数组（但有时更明确的做法是 np.zeros((height, width), dtype=...)）。此数组会被传给 rasterize 作为写入目标。
            out_arr = out.read(1)
            
            shapes = ((geom,value) for geom, value in zip(shapefile.get('geometry'), shapefile.get(field)))
            burned = features.rasterize(shapes=shapes, fill=0, out=out_arr, transform=out.transform)
            out.write_band(1, burned)




# 小麦产量优化量

In [3]:
#把 CSV 转 shapefile
os.chdir('E:/Jupyter/Data/')
csv_file = 'Wheat_XY_nozero.csv'  #当前文件路径
df = pd.read_csv(csv_file)
geometry = [Point(xyz) for xyz in zip(df['Lon'], df['Lat'],df['OY'])]   #变量名进行自主修改
gdf = gpd.GeoDataFrame(df, geometry=geometry)
gdf.set_crs(epsg=4326, inplace=True)
gdf.to_file('./shp/wheat/Wheat_YP_top10.shp')

In [5]:
#调用栅格化
in_shp = './shp/wheat/Wheat_YP_top10.shp'  # shp文件路径，这里引用的方式和上面双斜线的方式都可以
out_tif = './tiff/wheat/Wheat_YP_top10.tif'  # tiff文件路径
pixel_size = 0.01
field = 'OY'  #变量名进行自主修改
Polygon2Raster().polygon_to_raster(in_shp,out_tif,pixel_size,field)
print('finish')


finish


# 原版地图

In [3]:
import sys,os
import matplotlib
matplotlib.use('TkAgg')
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib as mpl
from matplotlib.colors import BoundaryNorm
import cartopy.io.shapereader as shpreader
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from cartopy.mpl.geoaxes import GeoAxes

# ================== 全局设置 ==================
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Arial'
mpl.rcParams['mathtext.it'] = 'Arial:italic'
mpl.rcParams['mathtext.bf'] = 'Arial:bold'
mpl.rcParams['axes.unicode_minus'] = False

# ================== 读取数据 ==================
os.chdir('E:/Jupyter/Data/')
tif_path = './tiff/wheat/Wheat_YP_top10.tif'

with rasterio.open(tif_path) as src:
    data = src.read(1).astype(float)
    transform = src.transform
    height, width = data.shape
    x_coords = transform.c + (np.arange(width) + 0.5) * transform.a
    y_coords = transform.f + (np.arange(height) + 0.5) * transform.e

# 把 0 视为无效值
data[data == 0] = np.nan
print(f"数据范围：最小值={np.nanmin(data)}, 最大值={np.nanmax(data)}")

# 构建网格
lon_grid, lat_grid = np.meshgrid(x_coords, y_coords)

# ================== 绘图函数 ==================
def plot_combined():
    lon_min_china, lon_max_china = 73, 135
    lat_min_china, lat_max_china = 18, 54

    lon_mask = (x_coords >= lon_min_china) & (x_coords <= lon_max_china)
    lat_mask = (y_coords >= lat_min_china) & (y_coords <= lat_max_china)

    lon_china = lon_grid[np.ix_(lat_mask, lon_mask)]
    lat_china = lat_grid[np.ix_(lat_mask, lon_mask)]
    data_china = data[np.ix_(lat_mask, lon_mask)]

    if np.all(np.isnan(data_china)):
        raise RuntimeError("裁剪后数据全部为空（全是 NaN）。请检查 tif 和裁剪范围。")

    fig = plt.figure(figsize=(10, 5))
    ax_map = fig.add_axes([0.1, 0.1, 0.8, 0.9], projection=ccrs.PlateCarree())
    ax_map.set_extent([lon_min_china, lon_max_china, lat_min_china, lat_max_china], crs=ccrs.PlateCarree())

  

    # === 中国边界 shapefile ===
    china_shp = r"E:\Jupyter\Data\weizhibiao\a5bc0-main\a5bc0-main\china_SHP\china_SHP\China_regions_Wheat.shp"
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_map.add_geometries([geom], ccrs.PlateCarree(),
                              edgecolor='black', facecolor='none', linewidth=1)

    # 设置经纬度刻度
    ax_map.set_xticks(np.arange(75, 136, 10), crs=ccrs.PlateCarree())
    ax_map.set_yticks(np.arange(20, 55, 10), crs=ccrs.PlateCarree())
    ax_map.xaxis.set_major_formatter(LongitudeFormatter())
    ax_map.yaxis.set_major_formatter(LatitudeFormatter())
    ax_map.tick_params(labelsize=12)

    # === 绘制数据 ===
    levels = [7000, 7500, 8000, 8500, 9000, 9500, 10000]
    cmap = plt.get_cmap('viridis_r')
    norm = BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend='both')

    contour = ax_map.contourf(
        lon_china, lat_china, data_china,
        levels=levels, cmap=cmap, norm=norm,
        transform=ccrs.PlateCarree(), extend='both'
    )

    # === 九段线（主图） ===
    nine_line_shp = r"E:\Jupyter\Data\weizhibiao\九段线\九段线\九段线.shp"
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_map.add_geometries([geom], ccrs.PlateCarree(),
                                  edgecolor='black', facecolor='none',
                                  linewidth=1.2, linestyle='--', zorder=5)
    except Exception as e:
        print("⚠️ 未加载九段线 shapefile：", e)

    # === 南海群岛插图（右下角，贴合主图边框） ===
    ax_inset = inset_axes(
        ax_map,
        width="18%", height="18%", #调整整个图的尺寸大小
        loc='lower right',
        borderpad=0,   #插图与主图边框无缝对齐
        axes_class=GeoAxes,
        axes_kwargs=dict(map_projection=ccrs.PlateCarree())
    )
    ax_inset.set_extent([94, 125, 7, 25], crs=ccrs.PlateCarree()) #插图的相对大小设置

    # 插图边界
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                edgecolor='black', facecolor='none', linewidth=0.7)

    # 插图数据
    lon_mask_inset = (x_coords >= 105) & (x_coords <= 125)
    lat_mask_inset = (y_coords >= 3) & (y_coords <= 25)
    lon_inset = lon_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    lat_inset = lat_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    data_inset = data[np.ix_(lat_mask_inset, lon_mask_inset)]

    if not np.all(np.isnan(data_inset)):
        ax_inset.contourf(
            lon_inset, lat_inset, data_inset,
            levels=levels, cmap=cmap, norm=norm,
            transform=ccrs.PlateCarree(), extend='both'
        )

    # 插图九段线
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                    edgecolor='black', facecolor='none', # facecolor='none'：九段线插图没有边框
                                    linewidth=1.0, linestyle='--', zorder=5)
    except Exception as e:
        print("插图未加载九段线 shapefile：", e)

    # 去掉插图坐标轴
    ax_inset.set_xticks([])
    ax_inset.set_yticks([])
    ax_inset.spines['geo'].set_visible(True)

    # === colorbar ===
    cbar = plt.colorbar(contour, ax=ax_map, orientation='vertical', shrink=0.7, pad=0.05) #改比例尺（horizontal是横向，vertical是竖向）
    #cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    cbar.set_label('Optimal yield (kg ha$^{-1}$)', fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    cbar.set_ticks(levels)




    # cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    # cbar.ax.tick_params(labelsize=12)
    # cbar.set_ticks(levels)

    #plt.savefig('./png/optimal yield potential_top10.png', dpi=600, bbox_inches='tight', pad_inches=0.01)
    plt.show()

# ================== 执行绘图 ==================
plot_combined()


数据范围：最小值=2677.5, 最大值=9646.49609375


E:\Anaconda\envs\DL\lib\site-packages\cartopy\mpl\geoaxes.py:395: UserWarning: The `map_projection` keyword argument is deprecated, use `projection` to instantiate a GeoAxes instead.
  warnings.warn("The `map_projection` keyword argument is "


# 小麦优化钾肥施用量

In [8]:
#调用栅格化
in_shp = './shp/wheat/Wheat_YP_top10.shp'  # shp文件路径，这里引用的方式和上面双斜线的方式都可以
out_tif = './tiff/wheat/Wheat_OKR_top10.tif'  # tiff文件路径
pixel_size = 0.01
field = 'OKR'  #变量名进行自主修改
Polygon2Raster().polygon_to_raster(in_shp,out_tif,pixel_size,field)
print('finish')

finish


In [5]:
import matplotlib
matplotlib.use('TkAgg')
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib as mpl
from matplotlib.colors import BoundaryNorm
import cartopy.io.shapereader as shpreader
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from cartopy.mpl.geoaxes import GeoAxes

# ================== 全局设置 ==================
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Arial'
mpl.rcParams['mathtext.it'] = 'Arial:italic'
mpl.rcParams['mathtext.bf'] = 'Arial:bold'
mpl.rcParams['axes.unicode_minus'] = False

# ================== 读取数据 ==================
tif_path = './tiff/wheat/Wheat_OKR_top10.tif'

with rasterio.open(tif_path) as src:
    data = src.read(1).astype(float)
    transform = src.transform
    height, width = data.shape
    x_coords = transform.c + (np.arange(width) + 0.5) * transform.a
    y_coords = transform.f + (np.arange(height) + 0.5) * transform.e

# 把 0 视为无效值
data[data == 0] = np.nan
print(f"数据范围：最小值={np.nanmin(data)}, 最大值={np.nanmax(data)}")

# 构建网格
lon_grid, lat_grid = np.meshgrid(x_coords, y_coords)

# ================== 绘图函数 ==================
def plot_combined():
    lon_min_china, lon_max_china = 73, 135
    lat_min_china, lat_max_china = 18, 54

    lon_mask = (x_coords >= lon_min_china) & (x_coords <= lon_max_china)
    lat_mask = (y_coords >= lat_min_china) & (y_coords <= lat_max_china)

    lon_china = lon_grid[np.ix_(lat_mask, lon_mask)]
    lat_china = lat_grid[np.ix_(lat_mask, lon_mask)]
    data_china = data[np.ix_(lat_mask, lon_mask)]

    if np.all(np.isnan(data_china)):
        raise RuntimeError("裁剪后数据全部为空（全是 NaN）。请检查 tif 和裁剪范围。")

    fig = plt.figure(figsize=(10, 5))
    ax_map = fig.add_axes([0.1, 0.1, 0.8, 0.9], projection=ccrs.PlateCarree())
    ax_map.set_extent([lon_min_china, lon_max_china, lat_min_china, lat_max_china], crs=ccrs.PlateCarree())

  

    # === 中国边界 shapefile ===
    china_shp = r"E:\Jupyter\Data\weizhibiao\a5bc0-main\a5bc0-main\china_SHP\china_SHP\China_regions_Wheat.shp"
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_map.add_geometries([geom], ccrs.PlateCarree(),
                              edgecolor='black', facecolor='none', linewidth=0.5)

    # 设置经纬度刻度
    ax_map.set_xticks(np.arange(75, 136, 10), crs=ccrs.PlateCarree())
    ax_map.set_yticks(np.arange(20, 55, 10), crs=ccrs.PlateCarree())
    ax_map.xaxis.set_major_formatter(LongitudeFormatter())
    ax_map.yaxis.set_major_formatter(LatitudeFormatter())
    ax_map.tick_params(labelsize=12)

    # === 绘制数据 ===
    # 不再定义离散的 levels，直接使用 Normalize
    #levels = [48, 54, 60, 66, 72, 78, 84]
    cmap = plt.get_cmap('viridis_r')
    #norm = BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend='both')
    # 使用 Normalize 进行连续性归一化
    norm = plt.Normalize(vmin=np.nanmin(data_china), vmax=np.nanmax(data_china))

    contour = ax_map.contourf(
        lon_china, lat_china, data_china, 
        cmap=cmap, norm=norm,
        transform=ccrs.PlateCarree(), extend='both'
    )


    # === 九段线（主图） ===
    nine_line_shp = r"E:\Jupyter\Data\weizhibiao\九段线\九段线\九段线.shp"
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_map.add_geometries([geom], ccrs.PlateCarree(),
                                  edgecolor='black', facecolor='none',
                                  linewidth=1.2, linestyle='--', zorder=5)
    except Exception as e:
        print("⚠️ 未加载九段线 shapefile：", e)

    # === 南海群岛插图（右下角，贴合主图边框） ===
    ax_inset = inset_axes(
        ax_map,
        width="18%", height="18%", #调整整个图的尺寸大小
        loc='lower right',
        borderpad=0,   #插图与主图边框无缝对齐
        axes_class=GeoAxes,
        axes_kwargs=dict(map_projection=ccrs.PlateCarree())
    )
    ax_inset.set_extent([94, 125, 7, 25], crs=ccrs.PlateCarree()) #插图的相对大小设置

    # 插图边界
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                edgecolor='black', facecolor='none', linewidth=0.7)

    # 插图数据
    lon_mask_inset = (x_coords >= 105) & (x_coords <= 125)
    lat_mask_inset = (y_coords >= 3) & (y_coords <= 25)
    lon_inset = lon_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    lat_inset = lat_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    data_inset = data[np.ix_(lat_mask_inset, lon_mask_inset)]

    if not np.all(np.isnan(data_inset)):
        ax_inset.contourf(
            lon_inset, lat_inset, data_inset,
            cmap=cmap, norm=norm,
            transform=ccrs.PlateCarree(), extend='both'
        )

    # 插图九段线
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                    edgecolor='black', facecolor='none', # facecolor='none'：九段线插图没有边框
                                    linewidth=1.0, linestyle='--', zorder=5)
    except Exception as e:
        print("插图未加载九段线 shapefile：", e)

    # 去掉插图坐标轴
    ax_inset.set_xticks([])
    ax_inset.set_yticks([])
    ax_inset.spines['geo'].set_visible(True)

    # === colorbar ===
    cbar = plt.colorbar(contour, ax=ax_map, orientation='vertical', shrink=0.7, pad=0.05) #改比例尺（horizontal是横向，vertical是竖向）
    cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    #cbar.set_label('Optimal yield (kg ha$^{-1}$)', fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    #cbar.set_ticks(levels)




    # cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    # cbar.ax.tick_params(labelsize=12)
    # cbar.set_ticks(levels)

    #plt.savefig('./png/optimal K rate2_top10.png', dpi=600, bbox_inches='tight', pad_inches=0.01)
    plt.show()

# ================== 执行绘图 ==================
plot_combined()

数据范围：最小值=39.0, 最大值=91.0


E:\Anaconda\envs\DL\lib\site-packages\cartopy\mpl\geoaxes.py:395: UserWarning: The `map_projection` keyword argument is deprecated, use `projection` to instantiate a GeoAxes instead.
  warnings.warn("The `map_projection` keyword argument is "


In [19]:
import matplotlib
matplotlib.use('TkAgg')
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib as mpl
from matplotlib.colors import BoundaryNorm
import cartopy.io.shapereader as shpreader
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from cartopy.mpl.geoaxes import GeoAxes

# ================== 全局设置 ==================
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Arial'
mpl.rcParams['mathtext.it'] = 'Arial:italic'
mpl.rcParams['mathtext.bf'] = 'Arial:bold'
mpl.rcParams['axes.unicode_minus'] = False

# ================== 读取数据 ==================
tif_path = './tiff/Maize_CY.tif'

with rasterio.open(tif_path) as src:
    data = src.read(1).astype(float)
    transform = src.transform
    height, width = data.shape
    x_coords = transform.c + (np.arange(width) + 0.5) * transform.a
    y_coords = transform.f + (np.arange(height) + 0.5) * transform.e

# 把 0 视为无效值
data[data == 0] = np.nan
print(f"数据范围：最小值={np.nanmin(data)}, 最大值={np.nanmax(data)}")

# 构建网格
lon_grid, lat_grid = np.meshgrid(x_coords, y_coords)

# ================== 绘图函数 ==================
def plot_combined():
    lon_min_china, lon_max_china = 73, 135
    lat_min_china, lat_max_china = 18, 54

    lon_mask = (x_coords >= lon_min_china) & (x_coords <= lon_max_china)
    lat_mask = (y_coords >= lat_min_china) & (y_coords <= lat_max_china)

    lon_china = lon_grid[np.ix_(lat_mask, lon_mask)]
    lat_china = lat_grid[np.ix_(lat_mask, lon_mask)]
    data_china = data[np.ix_(lat_mask, lon_mask)]

    if np.all(np.isnan(data_china)):
        raise RuntimeError("裁剪后数据全部为空（全是 NaN）。请检查 tif 和裁剪范围。")

    fig = plt.figure(figsize=(10, 5))
    ax_map = fig.add_axes([0.1, 0.1, 0.8, 0.9], projection=ccrs.PlateCarree())
    ax_map.set_extent([lon_min_china, lon_max_china, lat_min_china, lat_max_china], crs=ccrs.PlateCarree())

  

    # === 中国边界 shapefile ===
    china_shp = r"E:\Jupyter\Data\weizhibiao\a5bc0-main\a5bc0-main\china_SHP\china_SHP\国界_Project.shp"
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_map.add_geometries([geom], ccrs.PlateCarree(),
                              edgecolor='black', facecolor='none', linewidth=1)

    # 设置经纬度刻度
    ax_map.set_xticks(np.arange(75, 136, 10), crs=ccrs.PlateCarree())
    ax_map.set_yticks(np.arange(20, 55, 10), crs=ccrs.PlateCarree())
    ax_map.xaxis.set_major_formatter(LongitudeFormatter())
    ax_map.yaxis.set_major_formatter(LatitudeFormatter())
    ax_map.tick_params(labelsize=12)

    # === 绘制数据 ===
    levels = [9000, 10000, 11000, 12000, 13000, 14000, 15000]
    cmap = plt.get_cmap('viridis_r')
    norm = BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend='both')

    contour = ax_map.contourf(
        lon_china, lat_china, data_china,
        levels=levels, cmap=cmap, norm=norm,
        transform=ccrs.PlateCarree(), extend='both'
    )

    # === 九段线（主图） ===
    nine_line_shp = r"E:\Jupyter\Data\weizhibiao\九段线\九段线\九段线.shp"
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_map.add_geometries([geom], ccrs.PlateCarree(),
                                  edgecolor='black', facecolor='none',
                                  linewidth=1.2, linestyle='--', zorder=5)
    except Exception as e:
        print("⚠️ 未加载九段线 shapefile：", e)

    # === 南海群岛插图（右下角，贴合主图边框） ===
    ax_inset = inset_axes(
        ax_map,
        width="18%", height="18%", #调整整个图的尺寸大小
        loc='lower right',
        borderpad=0,   #插图与主图边框无缝对齐
        axes_class=GeoAxes,
        axes_kwargs=dict(map_projection=ccrs.PlateCarree())
    )
    ax_inset.set_extent([94, 125, 7, 25], crs=ccrs.PlateCarree()) #插图的相对大小设置

    # 插图边界
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                edgecolor='black', facecolor='none', linewidth=0.7)

    # 插图数据
    lon_mask_inset = (x_coords >= 105) & (x_coords <= 125)
    lat_mask_inset = (y_coords >= 3) & (y_coords <= 25)
    lon_inset = lon_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    lat_inset = lat_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    data_inset = data[np.ix_(lat_mask_inset, lon_mask_inset)]

    if not np.all(np.isnan(data_inset)):
        ax_inset.contourf(
            lon_inset, lat_inset, data_inset,
            levels=levels, cmap=cmap, norm=norm,
            transform=ccrs.PlateCarree(), extend='both'
        )

    # 插图九段线
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                    edgecolor='black', facecolor='none', # facecolor='none'：九段线插图没有边框
                                    linewidth=1.0, linestyle='--', zorder=5)
    except Exception as e:
        print("插图未加载九段线 shapefile：", e)

    # 去掉插图坐标轴
    ax_inset.set_xticks([])
    ax_inset.set_yticks([])
    ax_inset.spines['geo'].set_visible(True)

    # === colorbar ===
    cbar = plt.colorbar(contour, ax=ax_map, orientation='vertical', shrink=0.7, pad=0.05) #改比例尺（horizontal是横向，vertical是竖向）
    #cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    cbar.set_label('Current yield (kg ha$^{-1}$)', fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    cbar.set_ticks(levels)




    # cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    # cbar.ax.tick_params(labelsize=12)
    # cbar.set_ticks(levels)

    plt.savefig('./png/Current yield.png', dpi=600, bbox_inches='tight', pad_inches=0.01)
    plt.show()

# ================== 执行绘图 ==================
plot_combined()

数据范围：最小值=7422.7900390625, 最大值=17699.24609375


E:\Anaconda\envs\DL\lib\site-packages\cartopy\mpl\geoaxes.py:395: UserWarning: The `map_projection` keyword argument is deprecated, use `projection` to instantiate a GeoAxes instead.
  warnings.warn("The `map_projection` keyword argument is "


# 增产率

In [20]:
import matplotlib
matplotlib.use('TkAgg')
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib as mpl
from matplotlib.colors import BoundaryNorm
import cartopy.io.shapereader as shpreader
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from cartopy.mpl.geoaxes import GeoAxes

# ================== 全局设置 ==================
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Arial'
mpl.rcParams['mathtext.it'] = 'Arial:italic'
mpl.rcParams['mathtext.bf'] = 'Arial:bold'
mpl.rcParams['axes.unicode_minus'] = False

# ================== 读取数据 ==================
tif_path = './tiff/Maize_IR_top10.tif'

with rasterio.open(tif_path) as src:
    data = src.read(1).astype(float)
    transform = src.transform
    height, width = data.shape
    x_coords = transform.c + (np.arange(width) + 0.5) * transform.a
    y_coords = transform.f + (np.arange(height) + 0.5) * transform.e

# 把 0 视为无效值
data[data == 0] = np.nan
print(f"数据范围：最小值={np.nanmin(data)}, 最大值={np.nanmax(data)}")

# 构建网格
lon_grid, lat_grid = np.meshgrid(x_coords, y_coords)

# ================== 绘图函数 ==================
def plot_combined():
    lon_min_china, lon_max_china = 73, 135
    lat_min_china, lat_max_china = 18, 54

    lon_mask = (x_coords >= lon_min_china) & (x_coords <= lon_max_china)
    lat_mask = (y_coords >= lat_min_china) & (y_coords <= lat_max_china)

    lon_china = lon_grid[np.ix_(lat_mask, lon_mask)]
    lat_china = lat_grid[np.ix_(lat_mask, lon_mask)]
    data_china = data[np.ix_(lat_mask, lon_mask)]

    if np.all(np.isnan(data_china)):
        raise RuntimeError("裁剪后数据全部为空（全是 NaN）。请检查 tif 和裁剪范围。")

    fig = plt.figure(figsize=(10, 5))
    ax_map = fig.add_axes([0.1, 0.1, 0.8, 0.9], projection=ccrs.PlateCarree())
    ax_map.set_extent([lon_min_china, lon_max_china, lat_min_china, lat_max_china], crs=ccrs.PlateCarree())

  

    # === 中国边界 shapefile ===
    china_shp = r"E:\Jupyter\Data\weizhibiao\a5bc0-main\a5bc0-main\china_SHP\china_SHP\国界_Project.shp"
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_map.add_geometries([geom], ccrs.PlateCarree(),
                              edgecolor='black', facecolor='none', linewidth=1)

    # 设置经纬度刻度
    ax_map.set_xticks(np.arange(75, 136, 10), crs=ccrs.PlateCarree())
    ax_map.set_yticks(np.arange(20, 55, 10), crs=ccrs.PlateCarree())
    ax_map.xaxis.set_major_formatter(LongitudeFormatter())
    ax_map.yaxis.set_major_formatter(LatitudeFormatter())
    ax_map.tick_params(labelsize=12)

    # === 绘制数据 ===
    #levels = [10, 20, 30, 40]
    cmap = plt.get_cmap('viridis_r')
    #norm = BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend='both')
    # 使用 Normalize 进行连续性归一化
    norm = plt.Normalize(vmin=np.nanmin(data_china), vmax=np.nanmax(data_china))

    contour = ax_map.contourf(
        lon_china, lat_china, data_china,
        cmap=cmap, norm=norm,
        transform=ccrs.PlateCarree(), extend='both'
    )

    # === 九段线（主图） ===
    nine_line_shp = r"E:\Jupyter\Data\weizhibiao\九段线\九段线\九段线.shp"
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_map.add_geometries([geom], ccrs.PlateCarree(),
                                  edgecolor='black', facecolor='none',
                                  linewidth=1.2, linestyle='--', zorder=5)
    except Exception as e:
        print("⚠️ 未加载九段线 shapefile：", e)

    # === 南海群岛插图（右下角，贴合主图边框） ===
    ax_inset = inset_axes(
        ax_map,
        width="18%", height="18%", #调整整个图的尺寸大小
        loc='lower right',
        borderpad=0,   #插图与主图边框无缝对齐
        axes_class=GeoAxes,
        axes_kwargs=dict(map_projection=ccrs.PlateCarree())
    )
    ax_inset.set_extent([94, 125, 7, 25], crs=ccrs.PlateCarree()) #插图的相对大小设置

    # 插图边界
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                edgecolor='black', facecolor='none', linewidth=0.7)

    # 插图数据
    lon_mask_inset = (x_coords >= 105) & (x_coords <= 125)
    lat_mask_inset = (y_coords >= 3) & (y_coords <= 25)
    lon_inset = lon_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    lat_inset = lat_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    data_inset = data[np.ix_(lat_mask_inset, lon_mask_inset)]

    if not np.all(np.isnan(data_inset)):
        ax_inset.contourf(
            lon_inset, lat_inset, data_inset,
            cmap=cmap, norm=norm,
            transform=ccrs.PlateCarree(), extend='both'
        )

    # 插图九段线
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                    edgecolor='black', facecolor='none', # facecolor='none'：九段线插图没有边框
                                    linewidth=1.0, linestyle='--', zorder=5)
    except Exception as e:
        print("插图未加载九段线 shapefile：", e)

    # 去掉插图坐标轴
    ax_inset.set_xticks([])
    ax_inset.set_yticks([])
    ax_inset.spines['geo'].set_visible(True)

    # === colorbar ===
    cbar = plt.colorbar(contour, ax=ax_map, orientation='vertical', shrink=0.7, pad=0.05) #改比例尺（horizontal是横向，vertical是竖向）
    #cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    cbar.set_label('Increase rate (%)', fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    #cbar.set_ticks(levels)




    # cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    # cbar.ax.tick_params(labelsize=12)
    # cbar.set_ticks(levels)

    #plt.savefig('./png/Increase rate_top10.png', dpi=600, bbox_inches='tight', pad_inches=0.01)
    plt.show()

# ================== 执行绘图 ==================
plot_combined()

数据范围：最小值=12.002163887023926, 最大值=46.396629333496094


E:\Anaconda\envs\DL\lib\site-packages\cartopy\mpl\geoaxes.py:395: UserWarning: The `map_projection` keyword argument is deprecated, use `projection` to instantiate a GeoAxes instead.
  warnings.warn("The `map_projection` keyword argument is "


# 展示优化管理措施

In [9]:
import geopandas as gpd
import pandas as pd
import numpy as np
from rasterio.features import rasterize
from rasterio.transform import from_origin
import rasterio

# 1. 读取数据并查看
os.chdir('E:/Jupyter/Data/')
in_shp = './shp/Maize_YP_top10.shp'  # shp文件路径，这里引用的方式和上面双斜线的方式都可以
gdf = gpd.read_file(in_shp)
print("字段信息:")
print(gdf.dtypes)
print(f"\n'{field}'字段的唯一值:")
print(gdf['Management'].unique())

# 2. 手动创建编码字段
# 方法A: 使用pandas的factorize
gdf['Manageme_1'] = pd.factorize(gdf['Management'])[0] + 1
print("\n编码映射:")
for i, val in enumerate(gdf['Management'].unique()):
    print(f"{val} -> {i+1}")

# 3. 保存临时文件
temp_shp = './shp/Maize_YP_top10_encoded.shp'
gdf.to_file(temp_shp)

# 4. 使用编码后的字段    
out_tif = './tiff/Maize_manageemnt_top10.tif'  # tiff文件路径
pixel_size = 0.01
field = 'Manageme_1'  #变量名进行自主修改
Polygon2Raster().polygon_to_raster(temp_shp,out_tif,pixel_size,field)
print('finish')

字段信息:
Lon            float64
Lat            float64
optimal_K        int64
optimal_Y      float64
current        float64
increase       float64
increase_r     float64
Management      object
ID               int64
pre            float64
tem            float64
Maize_TEM      float64
Maize_PRE      float64
Maize_PET      float64
BD             float64
sand             int64
silt             int64
clay             int64
pH             float64
SOC            float64
TN             float64
AP             float64
AK             float64
geometry      geometry
dtype: object

'Management_1'字段的唯一值:
['I0_D2_T1_A1_Ty1_S0' 'I1_D1_T1_A1_Ty0_S1' 'I1_D0_T1_A1_Ty1_S0'
 'I0_D2_T1_A1_Ty0_S0' 'I1_D1_T1_A1_Ty1_S1' 'I1_D2_T1_A0_Ty0_S0'
 'I0_D1_T1_A1_Ty0_S0' 'I1_D0_T1_A1_Ty1_S1' 'I0_D2_T2_A1_Ty1_S0'
 'I0_D2_T2_A1_Ty0_S0' 'I1_D2_T1_A0_Ty0_S1' 'I1_D2_T2_A1_Ty0_S1'
 'I1_D1_T2_A1_Ty0_S1' 'I1_D1_T1_A1_Ty1_S0' 'I1_D2_T1_A1_Ty0_S1'
 'I1_D2_T1_A1_Ty0_S0' 'I1_D2_T1_A1_Ty1_S1' 'I1_D0_T1_A1_Ty0_S0'
 'I1_D1_T1_A1_Ty0_S0'

In [13]:
import matplotlib
matplotlib.use('TkAgg')
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib as mpl
from matplotlib.colors import BoundaryNorm
import cartopy.io.shapereader as shpreader
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from cartopy.mpl.geoaxes import GeoAxes

# ================== 全局设置 ==================
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Arial'
mpl.rcParams['mathtext.it'] = 'Arial:italic'
mpl.rcParams['mathtext.bf'] = 'Arial:bold'
mpl.rcParams['axes.unicode_minus'] = False

# ================== 读取数据 ==================
tif_path = './tiff/Maize_manageemnt_top10.tif'

with rasterio.open(tif_path) as src:
    data = src.read(1).astype(float)
    transform = src.transform
    height, width = data.shape
    x_coords = transform.c + (np.arange(width) + 0.5) * transform.a
    y_coords = transform.f + (np.arange(height) + 0.5) * transform.e

# 把 0 视为无效值
data[data == 0] = np.nan
print(f"数据范围：最小值={np.nanmin(data)}, 最大值={np.nanmax(data)}")

# 构建网格
lon_grid, lat_grid = np.meshgrid(x_coords, y_coords)

# ================== 绘图函数 ==================
def plot_combined():
    lon_min_china, lon_max_china = 73, 135
    lat_min_china, lat_max_china = 18, 54

    lon_mask = (x_coords >= lon_min_china) & (x_coords <= lon_max_china)
    lat_mask = (y_coords >= lat_min_china) & (y_coords <= lat_max_china)

    lon_china = lon_grid[np.ix_(lat_mask, lon_mask)]
    lat_china = lat_grid[np.ix_(lat_mask, lon_mask)]
    data_china = data[np.ix_(lat_mask, lon_mask)]

    if np.all(np.isnan(data_china)):
        raise RuntimeError("裁剪后数据全部为空（全是 NaN）。请检查 tif 和裁剪范围。")

    fig = plt.figure(figsize=(10, 5))
    ax_map = fig.add_axes([0.1, 0.1, 0.8, 0.9], projection=ccrs.PlateCarree())
    ax_map.set_extent([lon_min_china, lon_max_china, lat_min_china, lat_max_china], crs=ccrs.PlateCarree())

  

    # === 中国边界 shapefile ===
    china_shp = r"E:\Jupyter\Data\weizhibiao\a5bc0-main\a5bc0-main\china_SHP\china_SHP\国界_Project.shp"
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_map.add_geometries([geom], ccrs.PlateCarree(),
                              edgecolor='black', facecolor='none', linewidth=1)

    # 设置经纬度刻度
    ax_map.set_xticks(np.arange(75, 136, 10), crs=ccrs.PlateCarree())
    ax_map.set_yticks(np.arange(20, 55, 10), crs=ccrs.PlateCarree())
    ax_map.xaxis.set_major_formatter(LongitudeFormatter())
    ax_map.yaxis.set_major_formatter(LatitudeFormatter())
    ax_map.tick_params(labelsize=12)

    # === 绘制数据 ===
    #levels = [9000, 10000, 11000, 12000, 13000, 14000, 15000]
    cmap = plt.get_cmap('inferno')
    norm = plt.Normalize(vmin=np.nanmin(data_china), vmax=np.nanmax(data_china))

    contour = ax_map.contourf(
        lon_china, lat_china, data_china,
        cmap=cmap, norm=norm,
        transform=ccrs.PlateCarree(), extend='both'
    )

    # === 九段线（主图） ===
    nine_line_shp = r"E:\Jupyter\Data\weizhibiao\九段线\九段线\九段线.shp"
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_map.add_geometries([geom], ccrs.PlateCarree(),
                                  edgecolor='black', facecolor='none',
                                  linewidth=1.2, linestyle='--', zorder=5)
    except Exception as e:
        print("⚠️ 未加载九段线 shapefile：", e)

    # === 南海群岛插图（右下角，贴合主图边框） ===
    ax_inset = inset_axes(
        ax_map,
        width="18%", height="18%", #调整整个图的尺寸大小
        loc='lower right',
        borderpad=0,   #插图与主图边框无缝对齐
        axes_class=GeoAxes,
        axes_kwargs=dict(map_projection=ccrs.PlateCarree())
    )
    ax_inset.set_extent([94, 125, 7, 25], crs=ccrs.PlateCarree()) #插图的相对大小设置

    # 插图边界
    reader = shpreader.Reader(china_shp)
    for geom in reader.geometries():
        ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                edgecolor='black', facecolor='none', linewidth=0.7)

    # 插图数据
    lon_mask_inset = (x_coords >= 105) & (x_coords <= 125)
    lat_mask_inset = (y_coords >= 3) & (y_coords <= 25)
    lon_inset = lon_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    lat_inset = lat_grid[np.ix_(lat_mask_inset, lon_mask_inset)]
    data_inset = data[np.ix_(lat_mask_inset, lon_mask_inset)]

    if not np.all(np.isnan(data_inset)):
        ax_inset.contourf(
            lon_inset, lat_inset, data_inset,
            cmap=cmap, norm=norm,
            transform=ccrs.PlateCarree(), extend='both'
        )

    # 插图九段线
    try:
        reader = shpreader.Reader(nine_line_shp)
        for geom in reader.geometries():
            ax_inset.add_geometries([geom], ccrs.PlateCarree(),
                                    edgecolor='black', facecolor='none', # facecolor='none'：九段线插图没有边框
                                    linewidth=1.0, linestyle='--', zorder=5)
    except Exception as e:
        print("插图未加载九段线 shapefile：", e)

    # 去掉插图坐标轴
    ax_inset.set_xticks([])
    ax_inset.set_yticks([])
    ax_inset.spines['geo'].set_visible(True)

    # === colorbar ===
    cbar = plt.colorbar(contour, ax=ax_map, orientation='vertical', shrink=0.7, pad=0.05) #改比例尺（horizontal是横向，vertical是竖向）
    #cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    cbar.set_label('Best management package', fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    #cbar.set_ticks(levels)




    # cbar.set_label('Optimal K$_2$O rate', fontsize=14)
    # cbar.ax.tick_params(labelsize=12)
    # cbar.set_ticks(levels)

    #plt.savefig('./png/Current yield.png', dpi=600, bbox_inches='tight', pad_inches=0.01)
    plt.show()

# ================== 执行绘图 ==================
plot_combined()

数据范围：最小值=1.0, 最大值=47.0


E:\Anaconda\envs\DL\lib\site-packages\cartopy\mpl\geoaxes.py:395: UserWarning: The `map_projection` keyword argument is deprecated, use `projection` to instantiate a GeoAxes instead.
  warnings.warn("The `map_projection` keyword argument is "
